# 🛡️ Phase 6: Independent Evaluation Environment on Google Colab

Môi trường thực thi độc lập này đóng vai trò nạp mô hình đã fine-tune từ Hugging Face Hub (`thong7d/vihsd-phobert-hate-speech`) và chạy đánh giá (Phase 6) trên tập dữ liệu kiểm tra của ViHSD. 

### ✨ Đặc điểm thiết kế:
1. **Giải phóng Drive (Local-first)**: Không cần liên kết Google Drive, kết quả được tạo ngay tại bộ nhớ đệm và nén gửi về trình duyệt của bạn.
2. **Tương thích hoàn toàn (Tokenizer Sync)**: Sử dụng tách từ (`use_word_segmentation=True`) đồng bộ hoàn toàn với tokenizer của PhoBERT để đạt độ chính xác cao nhất.
3. **Tách biệt cấu hình (Modularity)**: Truyền trực tiếp repo ID thông qua CLI argument `--hf_repo_id` mà không làm thay đổi các file config huấn luyện.

### 🚀 Bước 1: Cấu hình môi trường và Tải mã nguồn

In [ ]:
# Cấu hình thư mục cache cho Hugging Face
import os, sys
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets_cache"

REPO_URL  = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    print(f"📥 Đang clone repository từ: {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_PATH}")
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    os.system(f"cd {REPO_PATH} && git pull origin main")

print("✅ Tải mã nguồn thành công!")

### 📦 Bước 2: Cài đặt thư viện dependencies

In [ ]:
print("📥 Đang cài đặt thư viện dependencies từ requirements.txt...")
os.system(f"pip install -q -r {REPO_PATH}/requirements.txt")

print("📥 Đang cài đặt thư viện tách từ tiếng Việt underthesea...")
os.system("pip install -q underthesea")

print("📥 Đang cài đặt dự án ở chế độ phát triển (editable mode)...")
os.system(f"pip install -q -e {REPO_PATH}")

if f"{REPO_PATH}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_PATH}/src")

print("✅ Cài đặt môi trường thành công!")

### 📊 Bước 3: Tải dữ liệu và Tiền xử lý tập Test (PhoBERT Tokenizer Sync)

In [ ]:
from src.data.download import download_and_extract
from src.data.preprocessing import process_split

print("📥 Đang tải dữ liệu gốc từ GitHub...")
download_and_extract(f"{REPO_PATH}/configs/paths.yaml")

print("⚙️ Đang thực hiện tiền xử lý tập test (giữ nguyên use_word_segmentation=True)...")
process_split(
    input_path=f"{REPO_PATH}/data/raw/vihsd/test.csv",
    output_path=f"{REPO_PATH}/data/processed/test.parquet",
    use_word_segmentation=True
)

print("✅ Tiền xử lý dữ liệu kiểm tra thành công! Tập test đã lưu dưới dạng Parquet.")

### 🧠 Bước 4: Thực hiện đánh giá mô hình qua CLI

In [ ]:
print("🚀 Bắt đầu quá trình đánh giá mô hình thong7d/vihsd-phobert-hate-speech...")
# Đánh giá mô hình nạp từ HF Hub và ghi đè tách từ là True
!python -m src.evaluation.evaluate --config {REPO_PATH}/configs/train.yaml --hf_repo_id thong7d/vihsd-phobert-hate-speech --use_word_segmentation True

### 📥 Bước 5: Nén và Tải kết quả về máy cục bộ

In [ ]:
import shutil
from google.colab import files

print("📦 Đang nén thư mục kết quả đánh giá (results/)...")
shutil.make_archive("/content/evaluation_results", "zip", f"{REPO_PATH}/results")

print("📥 Đang tải file evaluation_results.zip về máy cục bộ của bạn...")
files.download("/content/evaluation_results.zip")
print("✅ Đã kích hoạt hộp thoại tải xuống thành công!")

### 📝 Bước 6: Hiển thị báo cáo đánh giá trực tiếp

In [ ]:
from IPython.display import Markdown, display

report_path = f"{REPO_PATH}/results/evaluation_report.md"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        content = f.read()
    print("\n" + "="*40 + " BÁO CÁO ĐÁNH GIÁ CHI TIẾT " + "="*40 + "\n")
    display(Markdown(content))
else:
    print("⚠️ Không tìm thấy tệp báo cáo evaluation_report.md!")